<style>
.jp-Notebook, .jp-Cell, body.jp-Notebook, .cell, div#notebook, .notebook {
  background-color: #0f1115 !important;
}
.jp-RenderedMarkdown, .rendered_html, .jp-RenderedHTMLCommon {
  color: #e6e6e6 !important; font-size: 16px; line-height: 1.6;
}
.jp-RenderedMarkdown h1, .jp-RenderedMarkdown h2, .jp-RenderedMarkdown h3,
.rendered_html h1, .rendered_html h2, .rendered_html h3 { color: #8ec7ff !important; }
.jp-RenderedMarkdown h1, .rendered_html h1 {
  border-bottom: 2px solid #2b4a6f; padding-bottom: .2em;
}
.jp-RenderedMarkdown a, .rendered_html a { color: #7fd1c1 !important; }
.jp-RenderedMarkdown code, .rendered_html code {
  background: #1b2430 !important; color: #ffd479 !important;
  padding: 1px 5px; border-radius: 4px;
}
.jp-RenderedMarkdown pre, .rendered_html pre {
  background: #1b2430 !important; color: #e6e6e6 !important;
  border-left: 3px solid #2b4a6f; padding: .6em .9em; border-radius: 6px;
}
.jp-RenderedMarkdown blockquote, .rendered_html blockquote {
  border-left: 4px solid #7fd1c1; color: #bcd; background: #141a22;
  padding: .3em 1em; border-radius: 0 6px 6px 0;
}
.jp-RenderedMarkdown table, .rendered_html table { color: #e6e6e6; }
.jp-RenderedMarkdown th, .rendered_html th { background: #1b2430 !important; }
</style>


# NB17b · Las reglas de las pendientes

**Parte 2 · Las matemáticas del aprendizaje — Lección intermedia (entre el NB17 y el NB18)**

> En el NB16 aprendiste a **calcular** una pendiente con el ordenador: mirar la función un poquito a la izquierda y un poquito a la derecha, y dividir. Funciona siempre, pero tiene dos pegas: hay que calcular la función **dos veces** por cada pendiente (con millones de ruedecillas, eso es lentísimo, NB17), y no te dice nada de **por qué** la pendiente vale lo que vale.

Los matemáticos descubrieron hace más de 300 años unas pocas **reglas** que dan la pendiente **exacta** de casi cualquier función, con una fórmula, sin calcular nada dos veces. En el NB16 viste la primera: la pendiente de x² es 2·x. Hoy veremos todas las que necesita el curso, **una a una**, y cada una la **comprobaremos** con la pendiente numérica del NB16, para que no tengas que creerte nada.

Al terminar, sabrás calcular a mano la pendiente de funciones como (x − 3)², e^(−x²) o ln(x² + 1). Y entenderás de dónde salen las fórmulas que usaremos en el NB18 (la pendiente del error), en el NB19 (la retropropagación) y en el NB29 (cómo aprende un robot por refuerzo).


In [1]:
import numpy as np
import matplotlib.pyplot as plt

def pendiente(f, x, h=1e-5):
    # La pendiente numérica centrada del NB16: nuestra "verdad" para comprobar las reglas.
    return (f(x + h) - f(x - h)) / (2 * h)

def comprobar(f, regla, puntos=(-1.5, 0.3, 2.0)):
    # Compara la pendiente numérica con la que da una regla, en varios puntos.
    for x in puntos:
        print(f"  x = {x:+.1f}: numérica {pendiente(f, x):+.5f} | regla {regla(x):+.5f}")

(Dos funciones de ayuda. `pendiente` es la del NB16, con un valor por defecto para h (NB11b). `comprobar` recibe una función y una "regla" (otra función, que da la pendiente según la fórmula que estemos probando), y las compara en tres puntos. Si las columnas coinciden, la regla es correcta.)

## 1 · Notación

La pendiente de una función f en cada punto es, a su vez, **otra función** (NB16: la pendiente de x² es la función 2·x). Se le llama la **derivada** de f, y se escribe de dos formas:

- **f'(x)** (se lee "f prima de x"): la derivada de f.
- **df/dx** (se lee "de efe de equis"): "lo que cambia f cuando cambia x, un poquito". Recuerda la definición del NB16: un cambio pequeñito de f dividido por un cambio pequeñito de x.

Las dos significan lo mismo. La segunda es útil cuando hay varias letras, porque dice **respecto a qué** letra se mide la pendiente.


## 2 · Las reglas fáciles

### Una constante: pendiente 0

Si f(x) = 5 (una línea horizontal), la función no sube ni baja nunca: **su pendiente es 0** en todas partes.

### Una recta: su inclinación

Si f(x) = m·x + b (una recta, NB04b), su pendiente es **m** en todas partes: la recta sube m por cada unidad que avanzas, siempre igual. El `+ b` (que solo la sube o la baja, NB15b) no cambia la pendiente.

### Las potencias

La regla que viste en el NB16 para x² es un caso de una regla general para **cualquier potencia**:

```
   la pendiente de xⁿ es  n · xⁿ⁻¹         ("baja el exponente delante y réstale 1")
```

- x² → 2·x¹ = **2·x** (la del NB16).
- x³ → **3·x²**.
- x¹ = x → 1·x⁰ = **1** (la recta de pendiente 1, ✓).

Y funciona también con exponentes negativos y fraccionarios (NB03b):

- 1/x = x⁻¹ → −1·x⁻² = **−1/x²**.
- √x = x^0,5 → 0,5·x^(−0,5) = **1 / (2·√x)**.

Comprobémoslas:


In [2]:
print("x³:")
comprobar(lambda x: x ** 3, lambda x: 3 * x ** 2)
print("1/x:")
comprobar(lambda x: 1 / x, lambda x: -1 / x ** 2)
print("√x:")
comprobar(lambda x: np.sqrt(x), lambda x: 1 / (2 * np.sqrt(x)), puntos=(0.5, 1.0, 4.0))

x³:
  x = -1.5: numérica +6.75000 | regla +6.75000
  x = +0.3: numérica +0.27000 | regla +0.27000
  x = +2.0: numérica +12.00000 | regla +12.00000
1/x:
  x = -1.5: numérica -0.44444 | regla -0.44444
  x = +0.3: numérica -11.11111 | regla -11.11111
  x = +2.0: numérica -0.25000 | regla -0.25000
√x:
  x = +0.5: numérica +0.70711 | regla +0.70711
  x = +1.0: numérica +0.50000 | regla +0.50000
  x = +4.0: numérica +0.25000 | regla +0.25000


Coinciden en todos los puntos (hasta la quinta cifra decimal: lo que queda es el pequeño error de la pendiente numérica, NB16). (Las `lambda` son funciones de una línea, que verás a fondo en el NB23: `lambda x: x ** 3` es lo mismo que `def f(x): return x ** 3`. Aquí sirven para escribir cada función y su regla en el sitio, sin ponerles nombre.)

### Un número multiplicando: se queda

Si multiplicas una función por un número, su pendiente se multiplica por ese número: **la pendiente de c·f(x) es c·f'(x)**. Tiene sentido: si una curva es el triple de alta (NB15b), sube el triple de deprisa. Así, la pendiente de 5·x² es 5·2·x = **10·x**, y la de −x² es **−2·x**.

### Una suma: suma de pendientes

**La pendiente de una suma es la suma de las pendientes**: (f + g)' = f' + g'. Si una cosa sube 3 por segundo y otra 2, juntas suben 5. Así se deriva cualquier polinomio, término a término:

```
   f(x) = 4·x³ − 2·x² + 7·x − 1
   f'(x) = 12·x² − 4·x + 7
```


In [3]:
comprobar(lambda x: 4 * x ** 3 - 2 * x ** 2 + 7 * x - 1, lambda x: 12 * x ** 2 - 4 * x + 7)

  x = -1.5: numérica +40.00000 | regla +40.00000
  x = +0.3: numérica +6.88000 | regla +6.88000
  x = +2.0: numérica +47.00000 | regla +47.00000


## 3 · La regla de la cadena: engranajes

### Una función dentro de otra

Muchas funciones son **una función dentro de otra**. Por ejemplo, (x − 3)²: primero se calcula **x − 3** (la de dentro) y después se eleva **al cuadrado** (la de fuera). ¿Cómo se calcula su pendiente?

Imagina dos **engranajes**: el primero (la función de dentro) gira **2 vueltas** por cada vuelta que le das; el segundo (la de fuera) gira **3 vueltas** por cada vuelta del primero. Si giras una vuelta, el segundo da 2 × 3 = **6** vueltas. Los efectos se **multiplican**:

```
   pendiente de f(g(x))  =  (pendiente de la de fuera, en g(x))  ×  (pendiente de la de dentro)
                         =  f'(g(x)) · g'(x)
```

Es la **regla de la cadena**, la regla más importante de este notebook: es la base de todo el aprendizaje de las redes neuronales (NB18, NB19).

### Paso a paso con (x − 3)²

1. La de **dentro**: g(x) = x − 3. Su pendiente es **1** (una recta).
2. La de **fuera**: el cuadrado, f(u) = u². Su pendiente es **2·u**, y hay que calcularla **en lo de dentro**, u = x − 3: **2·(x − 3)**.
3. Se multiplican: 2·(x − 3) · 1 = **2·(x − 3)**.

Y si la función es **−(x − 3)²** (la colina del NB17), el signo menos de fuera se queda delante (regla de la constante): **−2·(x − 3)**. Esta es la pendiente que el NB17 usaba en su ejercicio E1: ahora sabes de dónde sale.

### Cuando lo de dentro tiene pendiente distinta de 1

Con **(2·x + 1)²**, la de dentro (2·x + 1) tiene pendiente **2**. Así que: 2·(2·x + 1) · **2** = **4·(2·x + 1)**. Y una trampa muy común: con **(3 − x)²**, la de dentro, 3 − x, tiene pendiente **−1** (baja). Resultado: 2·(3 − x)·(−1) = **−2·(3 − x)**. ¡Olvidar ese −1 es el error más típico de la regla de la cadena!


In [4]:
print("(x − 3)²:")
comprobar(lambda x: (x - 3) ** 2, lambda x: 2 * (x - 3))
print("(2x + 1)²:")
comprobar(lambda x: (2 * x + 1) ** 2, lambda x: 4 * (2 * x + 1))
print("(3 − x)²  (¡cuidado con el −1!):")
comprobar(lambda x: (3 - x) ** 2, lambda x: -2 * (3 - x))

(x − 3)²:
  x = -1.5: numérica -9.00000 | regla -9.00000
  x = +0.3: numérica -5.40000 | regla -5.40000
  x = +2.0: numérica -2.00000 | regla -2.00000
(2x + 1)²:
  x = -1.5: numérica -8.00000 | regla -8.00000
  x = +0.3: numérica +6.40000 | regla +6.40000
  x = +2.0: numérica +20.00000 | regla +20.00000
(3 − x)²  (¡cuidado con el −1!):
  x = -1.5: numérica -9.00000 | regla -9.00000
  x = +0.3: numérica -5.40000 | regla -5.40000
  x = +2.0: numérica -2.00000 | regla -2.00000


¿Te has fijado en que (3 − x)² da las mismas pendientes que (x − 3)²? Es lógico: las dos son **la misma función** (un número y su opuesto tienen el mismo cuadrado, NB03b), así que tienen que tener la misma pendiente. Y efectivamente, −2·(3 − x) = 2·(x − 3). Sin el −1 de lo de dentro, te habría salido justo la pendiente cambiada de signo: un error que la comprobación numérica caza al instante.

### Cadenas más largas

Si hay tres funciones una dentro de otra, se multiplican **tres** pendientes (tres engranajes), y así sucesivamente. En una red neuronal hay **cientos** de funciones encadenadas, y la regla de la cadena permite calcular la pendiente de todas ellas multiplicando pendientes pequeñas, una detrás de otra. A eso se le llama **retropropagación** (NB18, NB19).


## 4 · La regla del producto: el rectángulo

¿Y si dos funciones se **multiplican**, como x² · x³ o x · e^x? La pendiente **no** es el producto de las pendientes. La regla es:

```
   pendiente de f·g  =  f'·g  +  f·g'
```

La intuición es un **rectángulo** de lados f y g, cuya área es f·g. Si los dos lados crecen un poquito, el área crece por dos sitios: una tira a lo largo (lo que crece f, por el otro lado g: f'·g) y otra tira a lo ancho (f por lo que crece g: f·g'). (Y una esquinita minúscula, que con cambios pequeñitos es despreciable.)

```
   ┌────────────── g ──────────────┬─┐
   │                               │ │  ← f·g' (crece g)
   f          área f·g             │ │
   │                               │ │
   ├───────────────────────────────┼─┤
   └─── f'·g (crece f) ────────────┴─┘
```

Ejemplo: x² · x³. Con la regla: 2·x · x³ + x² · 3·x² = 2·x⁴ + 3·x⁴ = **5·x⁴**. Y efectivamente, x² · x³ = x⁵, cuya pendiente es 5·x⁴ ✓.


In [5]:
comprobar(lambda x: x ** 2 * x ** 3, lambda x: 2 * x * x ** 3 + x ** 2 * 3 * x ** 2)

  x = -1.5: numérica +25.31250 | regla +25.31250
  x = +0.3: numérica +0.04050 | regla +0.04050
  x = +2.0: numérica +80.00000 | regla +80.00000


## 5 · La exponencial y el logaritmo

### e^x es su propia pendiente

La función exponencial e^x (NB15b) tiene una propiedad única, que es la razón de que e sea tan especial: **su pendiente es ella misma**.

```
   la pendiente de eˣ es eˣ
```

En cada punto, la curva sube exactamente tanto como lo alta que está. Es la definición matemática de "crecer en proporción a lo que ya hay" (NB15b). Con la regla de la cadena salen todas sus variantes:

- e^(k·x) → e^(k·x) · **k** (la de dentro, k·x, tiene pendiente k). Por ejemplo, e^(−t/τ) → **−(1/τ)·e^(−t/τ)**: el decaimiento del NB15b baja más deprisa al principio, cuando queda mucho.
- La **campana** e^(−x²) → e^(−x²) · (−2·x) = **−2·x·e^(−x²)**: pendiente 0 en el centro (la cima), positiva a la izquierda, negativa a la derecha.


In [6]:
print("eˣ:")
comprobar(np.exp, np.exp)
print("e^(−x²):")
comprobar(lambda x: np.exp(-x ** 2), lambda x: -2 * x * np.exp(-x ** 2))

eˣ:
  x = -1.5: numérica +0.22313 | regla +0.22313
  x = +0.3: numérica +1.34986 | regla +1.34986
  x = +2.0: numérica +7.38906 | regla +7.38906
e^(−x²):
  x = -1.5: numérica +0.31620 | regla +0.31620
  x = +0.3: numérica -0.54836 | regla -0.54836
  x = +2.0: numérica -0.07326 | regla -0.07326


### La pendiente del logaritmo es 1/x

Para el logaritmo natural (NB15b):

```
   la pendiente de ln(x) es  1/x
```

Para x grande, la pendiente es pequeña (el logaritmo crece cada vez más despacio); para x cercano a 0, enorme (cae en picado hacia −∞).

Y con la regla de la cadena sale una forma que aparecerá en el corazón del aprendizaje por refuerzo (NB29, NB33):

```
   la pendiente de ln(f(x)) es  f'(x) / f(x)          ("la pendiente de f, dividida por f")
```

Es decir: la pendiente del **logaritmo** de algo es la pendiente de ese algo **dividida por su valor**. Mide el cambio **relativo** (en proporción, como un porcentaje, NB03b), no el absoluto.


In [7]:
print("ln(x):")
comprobar(np.log, lambda x: 1 / x, puntos=(0.5, 1.0, 4.0))
print("ln(x² + 1)  →  2x / (x² + 1):")
comprobar(lambda x: np.log(x ** 2 + 1), lambda x: 2 * x / (x ** 2 + 1))

ln(x):
  x = +0.5: numérica +2.00000 | regla +2.00000
  x = +1.0: numérica +1.00000 | regla +1.00000
  x = +4.0: numérica +0.25000 | regla +0.25000
ln(x² + 1)  →  2x / (x² + 1):
  x = -1.5: numérica -0.92308 | regla -0.92308
  x = +0.3: numérica +0.55046 | regla +0.55046
  x = +2.0: numérica +0.80000 | regla +0.80000


### La rampa: 0 o 1

Y una muy sencilla que usaremos en el NB19: la **rampa** (ReLU, NB15b), max(0, x). A la izquierda del codo es plana (**pendiente 0**); a la derecha, una recta de inclinación 1 (**pendiente 1**). Justo en el codo, la pendiente no está bien definida (cambia de golpe), pero en la práctica se toma 0 o 1 y no pasa nada.


## 6 · La tabla de las reglas

| Función | Su pendiente | Ejemplo |
|---|---|---|
| c (constante) | 0 | 5 → 0 |
| m·x + b | m | 3·x + 1 → 3 |
| xⁿ | n·xⁿ⁻¹ | x³ → 3·x²;  1/x → −1/x² |
| c·f(x) | c·f'(x) | 5·x² → 10·x |
| f + g | f' + g' | x² + x → 2·x + 1 |
| f(g(x)) **(cadena)** | f'(g(x)) · g'(x) | (x − 3)² → 2·(x − 3) |
| f · g **(producto)** | f'·g + f·g' | x²·x³ → 5·x⁴ |
| eˣ | eˣ | e^(k·x) → k·e^(k·x) |
| ln(x) | 1/x | ln(f) → f'/f |
| rampa(x) | 0 (izquierda) o 1 (derecha) | |

Con estas diez reglas se pueden derivar a mano casi todas las funciones del curso. (Las de **seno** y **coseno**, que necesitarán los péndulos y los muelles, las verás cuando lleguen esas funciones, en el NB39b.)


## 7 · La pendiente de la pendiente: la segunda derivada

### Derivar dos veces

La derivada de una función es otra función... que también tiene pendiente. A la pendiente de la pendiente se le llama **segunda derivada**, y se escribe **f''(x)** ("f dos primas"). Por ejemplo, para x³: la primera derivada es 3·x², y la segunda, 6·x.

### Qué significa: la aceleración

Ya lo viste en el NB16 sin este nombre: la **velocidad** es la pendiente de la **posición**, y la **aceleración** es la pendiente de la velocidad. Así que **la aceleración es la segunda derivada de la posición**. En la caída libre (NB04b), la posición es h = 2 − ½·g·t²; su primera derivada (la velocidad) es −g·t, y su segunda (la aceleración), **−g**: constante, la gravedad. Toda la física de los robots (NB37 en adelante) se escribe con segundas derivadas: "la fuerza decide la **aceleración**".

### Cima o valle

La segunda derivada también dice **qué forma** tiene la curva:

- **f'' > 0**: la pendiente va **aumentando**: la curva se dobla hacia **arriba**, como un cuenco (un **valle**).
- **f'' < 0**: la pendiente va **disminuyendo**: se dobla hacia **abajo**, como una montaña (una **cima**).

En un punto donde la pendiente es 0 (NB17: en lo alto o en lo bajo), la segunda derivada te dice si es una cima (f'' < 0) o un valle (f'' > 0). En la colina −(x − 3)², f' = −2·(x − 3) se anula en x = 3, y f'' = **−2** < 0: es una cima ✓.

### El método de Newton (una idea para más adelante)

El ascenso por gradiente del NB17 da pasos de tamaño fijo (la tasa). Pero si conoces también la segunda derivada (cuánto se curva el terreno), puedes **calcular** el tamaño de paso ideal: donde el terreno se curva mucho, pasos cortos; donde casi no se curva, pasos largos. Esa es la idea del **método de Newton**, mucho más rápido cerca de la cima. Lo verás nombrado en el NB46 (para la cinemática inversa) y en el NB48 (uno de los "solucionadores" de MuJoCo usa segundas derivadas).


In [8]:
f = lambda x: x ** 3
f1 = lambda x: pendiente(f, x)              # primera derivada (numérica)
f2 = lambda x: pendiente(f1, x, h=1e-3)     # segunda derivada: la pendiente de la pendiente
for x in [-1.0, 0.5, 2.0]:
    print(f"x = {x:+.1f}: f'' numérica {f2(x):+.4f} | regla 6·x = {6 * x:+.4f}")

x = -1.0: f'' numérica -6.0000 | regla 6·x = -6.0000
x = +0.5: f'' numérica +3.0000 | regla 6·x = +3.0000
x = +2.0: f'' numérica +12.0000 | regla 6·x = +12.0000


(Para la segunda derivada numérica usamos un h más grande, 0,001: al derivar dos veces, los errores de los decimales se amplifican, NB16, y un h demasiado pequeño los haría visibles.)

## 8 · Derivadas parciales

En el NB17 viste las **derivadas parciales**: si una función depende de **varias** ruedecillas, f(x, y), su pendiente "respecto a x" se calcula moviendo **solo x** y dejando y quieta. Ahora puedes calcularlas con las reglas: **se deriva respecto a una letra tratando las demás como si fueran números fijos**. Se escribe con una "d" redondeada, **∂f/∂x**.

Por ejemplo, f(x, y) = x²·y + 3·y:

- **∂f/∂x**: y es un número fijo. x²·y es "un número (y) por x²" → 2·x·y. Y 3·y no tiene x: es una constante → 0. Total: **∂f/∂x = 2·x·y**.
- **∂f/∂y**: x es un número fijo. x²·y es "un número (x²) por y" → x². Y 3·y → 3. Total: **∂f/∂y = x² + 3**.

Juntas forman el **gradiente** (NB17): la flecha (2·x·y, x² + 3) que apunta hacia donde la función sube más deprisa.


In [9]:
f = lambda x, y: x ** 2 * y + 3 * y
x, y = 1.5, -2.0
print("∂f/∂x numérica:", pendiente(lambda t: f(t, y), x), "| regla 2·x·y:", 2 * x * y)
print("∂f/∂y numérica:", pendiente(lambda t: f(x, t), y), "| regla x² + 3:", x ** 2 + 3)

∂f/∂x numérica: -6.000000000128124 | regla 2·x·y: -6.0
∂f/∂y numérica: 5.2500000000677005 | regla x² + 3: 5.25


(El truco para la parcial numérica: `lambda t: f(t, y)` es una función de **una** sola letra, t, con y fija: justo "mover solo x". Es la "función envoltorio" del NB17.)


## 9 · Resumen de la lección

1. La **derivada** f'(x) (o df/dx) es la función pendiente. Las **reglas** la dan exacta, sin calcular la función dos veces.
2. Constante → 0; recta m·x + b → m; **xⁿ → n·xⁿ⁻¹** (vale para 1/x y √x); c·f → c·f'; suma → suma de pendientes.
3. **Regla de la cadena** (engranajes): f(g(x)) → f'(g(x))·g'(x). ¡No olvides la pendiente de lo de dentro (el −1 de (3 − x)²)! Base de la retropropagación.
4. **Producto** (el rectángulo): f·g → f'·g + f·g'.
5. **eˣ es su propia pendiente**; e^(k·x) → k·e^(k·x); campana e^(−x²) → −2·x·e^(−x²).
6. **ln(x) → 1/x**; **ln(f) → f'/f** (cambio relativo). Rampa → 0 o 1.
7. **Segunda derivada** f'': pendiente de la pendiente. La aceleración es la segunda derivada de la posición. f'' < 0 → cima; f'' > 0 → valle. Método de Newton.
8. **Derivadas parciales** ∂f/∂x: deriva respecto a una letra con las demás fijas. Juntas, el gradiente.

### Palabras nuevas de hoy

| Palabra | Qué significa |
|---|---|
| **Derivada, f'(x), df/dx** | La función que da la pendiente de f en cada punto. |
| **Regla de la cadena** | La pendiente de una función dentro de otra es el producto de sus pendientes. |
| **Regla del producto** | (f·g)' = f'·g + f·g'. |
| **Segunda derivada, f''(x)** | La pendiente de la pendiente: cómo se curva una función. |
| **Método de Newton** | Buscar cimas o valles usando también la segunda derivada. |
| **Derivada parcial, ∂f/∂x** | Pendiente respecto a una letra, con las demás fijas. |


## 10 · Ejercicios

Calcula cada pendiente **a mano** con las reglas, y después compruébala con `comprobar`.

**E1.** f(x) = 3·x⁴ − x² + 5.

**E2.** f(x) = (x² + 1)³. (Cadena: la de fuera es el cubo.)

**E3.** f(x) = (5 − 2·x)². ¡Cuidado con lo de dentro!

**E4.** f(x) = x · eˣ. (Producto.)

**E5.** f(x) = e^(−(x − 1)² / 2). (La campana del NB15b centrada en 1.)

**E6.** f(x) = ln(3·x). ¿Por qué sale lo mismo que la pendiente de ln(x)? (Pista: la regla del producto de los logaritmos, NB15b.)

**E7.** El error cuadrático de una predicción, como el que usarás en el NB18, es E(w) = (w·x − y)², donde w es la ruedecilla, y x e y son números fijos. Calcula dE/dw.

**E8.** Para f(x, y) = x·y² − 2·x, calcula ∂f/∂x y ∂f/∂y, y el gradiente en el punto (1, 2).


<details>
<summary>▶ Solución E1</summary>

Término a término: 3·4·x³ − 2·x + 0 = **12·x³ − 2·x**.

```python
comprobar(lambda x: 3 * x ** 4 - x ** 2 + 5, lambda x: 12 * x ** 3 - 2 * x)
```
</details>

<details>
<summary>▶ Solución E2</summary>

Fuera, el cubo: u³ → 3·u², en u = x² + 1: 3·(x² + 1)². Dentro, x² + 1 → 2·x. Multiplicando: **6·x·(x² + 1)²**.

```python
comprobar(lambda x: (x ** 2 + 1) ** 3, lambda x: 6 * x * (x ** 2 + 1) ** 2)
```
</details>

<details>
<summary>▶ Solución E3</summary>

Fuera: 2·(5 − 2·x). Dentro, 5 − 2·x tiene pendiente **−2**. Total: 2·(5 − 2·x)·(−2) = **−4·(5 − 2·x)**.

```python
comprobar(lambda x: (5 - 2 * x) ** 2, lambda x: -4 * (5 - 2 * x))
```
</details>

<details>
<summary>▶ Solución E4</summary>

f = x (pendiente 1), g = eˣ (pendiente eˣ). Producto: 1·eˣ + x·eˣ = **(1 + x)·eˣ**.

```python
comprobar(lambda x: x * np.exp(x), lambda x: (1 + x) * np.exp(x))
```
</details>

<details>
<summary>▶ Solución E5</summary>

Fuera, e^u → e^u. Dentro, u = −(x − 1)²/2, cuya pendiente es −2·(x − 1)/2 = −(x − 1). Total: **−(x − 1)·e^(−(x − 1)²/2)**. Vale 0 en x = 1 (la cima de la campana), como debe ser.

```python
comprobar(lambda x: np.exp(-(x - 1) ** 2 / 2), lambda x: -(x - 1) * np.exp(-(x - 1) ** 2 / 2))
```
</details>

<details>
<summary>▶ Solución E6</summary>

Con la cadena: (1/(3·x)) · 3 = **1/x**. Sale lo mismo que ln(x) porque ln(3·x) = ln(3) + ln(x) (NB15b), y ln(3) es una **constante** (pendiente 0): las dos funciones son la misma curva, solo desplazada hacia arriba (NB15b), así que tienen la misma pendiente en cada punto.

```python
comprobar(lambda x: np.log(3 * x), lambda x: 1 / x, puntos=(0.5, 1.0, 4.0))
```
</details>

<details>
<summary>▶ Solución E7</summary>

Fuera, el cuadrado: 2·(w·x − y). Dentro, w·x − y como función de **w** (x e y son números fijos) es una recta de pendiente **x**. Total: **dE/dw = 2·(w·x − y)·x**: "dos por el error por la entrada". En el NB18 encontrarás exactamente esta fórmula (con una media de muchos ejemplos), con los engranajes como explicación.

```python
x_dato, y_dato = 2.0, 3.0
comprobar(lambda w: (w * x_dato - y_dato) ** 2, lambda w: 2 * (w * x_dato - y_dato) * x_dato)
```
</details>

<details>
<summary>▶ Solución E8</summary>

- ∂f/∂x (y fija): y² − 2.
- ∂f/∂y (x fija): x · 2·y = 2·x·y.

En (1, 2): ∂f/∂x = 4 − 2 = **2**; ∂f/∂y = 2·1·2 = **4**. Gradiente: **(2, 4)**.

```python
f = lambda x, y: x * y ** 2 - 2 * x
print(pendiente(lambda t: f(t, 2.0), 1.0), pendiente(lambda t: f(1.0, t), 2.0))    # 2.0  4.0
```
</details>


## 11 · Posdata

Si algo no ha quedado claro, dime el **apartado** y la **frase exacta** y lo reescribo.

En el **NB18**, el aprendizaje **supervisado**: enseñar a una máquina con ejemplos resueltos. Usarás la regla de la cadena (el ejercicio E7 de hoy) para calcular de golpe, y de forma exacta, hacia dónde mover sus ruedecillas.
